In [1]:
"""ps5_solution.py  —  Complete solution for 'What News Drive 15‑Minute Stock Variation Return?'

Usage
-----
python ps5_solution.py --market /path/to/SPY_15min.csv \
                       --events /path/to/event_db.csv \
                       --outdir /path/to/output_dir \
                       --bootstrap 200 --block 20

Requirements
------------
pandas, numpy, matplotlib, statsmodels, scikit-learn, tqdm

Output
------
* explained_share.png       — time series plot of explained vs. unexplained variance
* variance_by_cat.png       — bar chart of Ω_k by news category
* regression_results.txt    — textual summary incl. Ω_k estimates & CI
"""
 
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LassoCV
from tqdm import tqdm

def parse_args():
    p = argparse.ArgumentParser(description='Variance‑decomposition of 15‑min SPY returns.')
    p.add_argument('--market', required=True, help='CSV with 15‑min prices (datetime, price)')
    p.add_argument('--events', required=True, help='CSV with timestamped news events incl. category')
    p.add_argument('--outdir', default='.', help='Output directory')
    p.add_argument('--bootstrap', type=int, default=0, help='Number of block bootstrap resamples (0=skip)')
    p.add_argument('--block', type=int, default=20, help='Block length (obs) for bootstrap')
    return p.parse_args()

def load_market(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    ts_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
    price_col = [c for c in df.columns if 'price' in c.lower() or 'close' in c.lower()][0]
    df[ts_col] = pd.to_datetime(df[ts_col], utc=True)
    df.set_index(ts_col, inplace=True)
    df = df.asfreq('15min')
    df[price_col] = df[price_col].interpolate()
    df['log_price'] = np.log(df[price_col])
    df['r'] = df['log_price'].diff()
    df['rv'] = df['r']**2
    df.dropna(inplace=True)
    return df[['r', 'rv']]

def load_events(path: str) -> pd.DataFrame:
    ev = pd.read_csv(path)
    ts_col = [c for c in ev.columns if 'date' in c.lower() or 'time' in c.lower()][0]
    ev[ts_col] = pd.to_datetime(ev[ts_col], utc=True)
    ev['bin'] = ev[ts_col].dt.floor('15min')
    cat_col_candidates = [c for c in ev.columns if 'type' in c.lower() or 'cat' in c.lower()]
    if not cat_col_candidates:
        raise ValueError("Missing category/type column in event file.")
    cat_col = cat_col_candidates[0]
    ev = ev[['bin', cat_col]].rename(columns={cat_col: 'category'})
    mapping = {
        'macro': 'growth', 'gdp': 'growth', 'nfp': 'growth', 'cpi': 'growth',
        'money': 'money', 'monetary': 'money', 'fed': 'money',
        'ecb': 'money', 'earnings': 'ad‑hoc', 'geopolitical': 'ad‑hoc', 'fiscal': 'fiscal'
    }
    ev['cat'] = ev['category'].str.lower().map(lambda x: next((v for k, v in mapping.items() if k in x), 'other'))
    return ev[['bin', 'cat']]

def build_design(df_market: pd.DataFrame, ev: pd.DataFrame) -> pd.DataFrame:
    cats = ['growth', 'money', 'fiscal', 'ad‑hoc', 'other']
    dummy = pd.get_dummies(ev['cat'])
    dummy['bin'] = ev['bin']
    dummy = dummy.groupby('bin').max()
    X = df_market.join(dummy, how='left').fillna(0)
    X['hour'] = X.index.hour
    X['weekday'] = X.index.dayofweek
    hour_dummies = pd.get_dummies(X['hour'], prefix='h', drop_first=True)
    wd_dummies = pd.get_dummies(X['weekday'], prefix='wd', drop_first=True)
    X = pd.concat([X, hour_dummies, wd_dummies], axis=1)
    for l in [1, 4, 8]:
        X[f'rv_lag{l}'] = X['rv'].shift(l)
    X.dropna(inplace=True)
    return X

def run_regression(X: pd.DataFrame):
    y = X['rv']
    feature_cols = X.columns.difference(['r', 'rv', 'hour', 'weekday'])
    X_feat = X[feature_cols]
    lassocv = LassoCV(cv=5, random_state=1, n_jobs=-1).fit(X_feat, y)
    keep = X_feat.columns[lassocv.coef_ != 0]
    X_sel = sm.add_constant(X_feat[keep])
    model = sm.OLS(y, X_sel).fit()
    return model, keep

def variance_decomp(model, X, keep):
    var_r = X['rv'].var()
    p_dict = {k: X[k].mean() for k in keep if k in ['growth', 'money', 'fiscal', 'ad‑hoc', 'other']}
    omega = {k: model.params.get(k, 0) * p / var_r for k, p in p_dict.items()}
    omega_total = sum(omega.values())
    return omega, omega_total

def stationary_bootstrap_indices(n, block_length):
    idx = []
    rng = np.random.default_rng()
    start = rng.integers(0, n)
    idx.append(start)
    for _ in range(1, n):
        if rng.random() < 1 / block_length:
            start = rng.integers(0, n)
        else:
            start = (start + 1) % n
        idx.append(start)
    return np.array(idx)

def bootstrap_omega(X, keep, B=200, block_length=20):
    n = len(X)
    y = X['rv'].values
    feature_cols = X.columns.difference(['r', 'rv', 'hour', 'weekday'])
    X_feat = X[feature_cols].values
    omegas = []
    for _ in tqdm(range(B), desc='Bootstrap'):
        idx = stationary_bootstrap_indices(n, block_length)
        y_b = y[idx]
        X_b = X_feat[idx, :]
        X_b = sm.add_constant(X_b)
        model_b = sm.OLS(y_b, X_b).fit()
        coef_map = dict(zip(['const'] + list(feature_cols), model_b.params))
        p_dict = {k: X[k].mean() for k in keep if k in ['growth', 'money', 'fiscal', 'ad‑hoc', 'other']}
        var_r = np.var(y_b)
        omega_b = [coef_map.get(k, 0) * p_dict.get(k, 0) / var_r if var_r > 0 else 0 for k in ['growth', 'money', 'fiscal', 'ad‑hoc', 'other']]
        omegas.append(omega_b)
    return np.array(omegas)

def make_plots(X, model, keep, omega, outdir):
    Path(outdir).mkdir(parents=True, exist_ok=True)
    pred = model.predict(sm.add_constant(X[keep]))
    explained = pred
    plt.figure(figsize=(10,4))
    plt.plot(explained.cumsum(), label='Explained (cum.)')
    plt.plot(X['rv'].cumsum(), label='Total RV (cum.)', alpha=0.7, linestyle='--')
    plt.title('Cumulative Explained vs. Total Realised Variance')
    plt.legend()
    plt.tight_layout()
    plt.savefig(Path(outdir)/'explained_share.png')
    plt.close()

    plt.figure(figsize=(6,4))
    cats = list(omega.keys())
    vals = [omega[c] for c in cats]
    plt.bar(cats, vals)
    plt.title('Variance Share Ω_k by News Category')
    plt.ylabel('Share')
    plt.tight_layout()
    plt.savefig(Path(outdir)/'variance_by_cat.png')
    plt.close()

def main():
    args = parse_args()
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    df_market = load_market(args.market)
    ev = load_events(args.events)
    X = build_design(df_market, ev)
    model, keep = run_regression(X)
    omega, omega_total = variance_decomp(model, X, keep)

    print('Ω_k:')
    for k, v in omega.items():
        print(f'{k:7s}: {v*100:5.2f} %')
    print(f'Σ News: {omega_total*100:.2f} %')

    if args.bootstrap:
        omegas_b = bootstrap_omega(X, keep, B=args.bootstrap, block_length=args.block)
        ci_lower = np.percentile(omegas_b, 2.5, axis=0)
        ci_upper = np.percentile(omegas_b, 97.5, axis=0)
        for i, cat in enumerate(['growth', 'money', 'fiscal', 'ad‑hoc', 'other']):
            print(f'CI {cat:7s}: [{ci_lower[i]*100:.2f} %, {ci_upper[i]*100:.2f} %]')

    with open(outdir/'regression_results.txt', 'w') as fh:
        fh.write(model.summary().as_text())
        fh.write('\\nVariance decomposition Ω_k:\\n')
        for k, v in omega.items():
            fh.write(f'{k}: {v}\\n')
        fh.write(f'Total: {omega_total}\\n')

    make_plots(X, model, list(keep), omega, outdir)

if __name__ == '__main__':
    main()


usage: ipykernel_launcher.py [-h] --market MARKET --events EVENTS
                             [--outdir OUTDIR] [--bootstrap BOOTSTRAP]
                             [--block BLOCK]
ipykernel_launcher.py: error: the following arguments are required: --market, --events


SystemExit: 2

/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
